# Cálculo del total nacional para junio de 2026

### 1. Para el total nacional calcule (a Junio de 2026): 
### - -población en edad de trabajar  
### - -población que no está en edad de trabajar 
### - -población económicamente activa 
### - -población económicamente inactiva 
### - -tasa de ocupación 
### - -tasa de desempleo  

In [0]:
#Leyendo la tabla de características generales del catálogo
cgse=spark.read.format("delta").table("workspace.electiva.cgse")

### -población en edad de trabajar

In [0]:
PET = ft[ft['PET']==1]['FEX_C18'].sum()
print(f'La población que está en edad de trabajar es de {PET:,.0f} personas')

###-población que no está en edad de trabajar

In [0]:
cgse = cgse.toPandas()
PT = cgse['FEX_C18'].sum()
PNET = PT - PET
print(f'La población que no está en edad de trabajar es de {PNET:,.0f} personas')

### -población económicamente activa 

In [0]:
PEA = ft[ft['FT']==1]['FEX_C18'].sum()
print(f'La población económicamente activa es de {PEA:,.0f} personas')

###-población económicamente inactiva 

In [0]:
PEI = PET - PEA
print(f'La población económicamente inactiva es de {PEI:,.0f} personas')

### -tasa de ocupación 

In [0]:
ocupados = spark.read.format("delta").table("workspace.electiva.ocupados").toPandas()
O = ocupados[ocupados['OCI']==1]['FEX_C18'].sum()
TO = O/PEA*100
TO = O/PET*100
print(f'El porcentaje de ocupados es de {TO:.2f}%')

### -tasa de desempleo 

In [0]:
D = desempleados[desempleados['DSI']==1]['FEX_C18'].sum()  # Cell 23
TD = D/PEA*100
print(f'El porcentaje de desempleados es de {TD:.2f}%')

### Conclusión primera pregunta

Para junio de 2026, la población total de Colombia se estima en aproximadamente 52.5 millones de personas, de las cuales 41.2 millones están en edad de trabajar (PET) y 11.3 millones no lo están. Esto significa que cerca del 78% de la población está en edad laboral.

Dentro de la PET, la población económicamente activa (PEA) alcanza 26.6 millones de personas, mientras que la población económicamente inactiva (PEI) suma 14.6 millones. Es decir, aproximadamente el 65% de quienes están en edad de trabajar participan activamente en el mercado laboral.

La tasa de ocupación se sitúa en 59.36%, lo que indica que poco más de la mitad de la población en edad de trabajar se encuentra ocupada. Por su parte, la tasa de desempleo se ubica en 8.03%, reflejando que de cada 100 personas económicamente activas, aproximadamente 8 se encuentran desempleadas.

En conjunto, estos indicadores muestran un mercado laboral con una participación activa considerable, pero con un porcentaje no despreciable de la población en edad de trabajar que permanece fuera de la fuerza laboral, ya sea por estudio, labores del hogar u otras razones.

### 2. Calcule las tasas de desempleo para hombres y mujeres en todos los departamentos de Colombia a Junio de 2026, ¿cuáles son los departamentos con mayores y menores tasas de desempleo?  

In [0]:
import pandas as pd

# Leyendo la tabla de características generales
ft = spark.read.format("delta").table("workspace.electiva.ft")
desempleados = spark.read.format("delta").table("workspace.electiva.desempleados")
cgse = spark.read.format("delta").table("workspace.electiva.cgse")

# Convertir el DataFrame de Spark a Pandas
ft = ft.toPandas()
desempleados = desempleados.toPandas()
cgse = cgse.toPandas()

# Primero hay que anexar la P3271 a la tabla de desempleados
desempleados = pd.merge(
    desempleados,
    cgse[['DIRECTORIO', 'SECUENCIA_P', 'ORDEN', 'P3271', 'P6040']],
    on=['DIRECTORIO', 'SECUENCIA_P', 'ORDEN'],
    how='left'
)

ft = pd.merge(
    ft,
    cgse[['DIRECTORIO', 'SECUENCIA_P', 'ORDEN', 'P3271', 'P6040']],
    on=['DIRECTORIO', 'SECUENCIA_P', 'ORDEN'],
    how='left'
)

In [0]:
# desempleados_dpto -> cantidad de desempleados por departamento
desempleados_dpto = desempleados[desempleados['DSI'] == 1].groupby(['P3271', 'DPTO'])['FEX_C18'].sum()

In [0]:
desempleados_dpto

In [0]:
# Cantidad de hombres y mujeres económicamente activos por departamento
pea_dpto_genero = ft[ft['FT'] == 1].groupby(['P3271', 'DPTO'])['FEX_C18'].sum()

In [0]:
tabla_desempleo = pd.DataFrame({'PEA': pea_dpto_genero,'DESEMPLEADOS': desempleados_dpto})

In [0]:
tabla_desempleo

In [0]:
tabla_desempleo['TD'] = (
    tabla_desempleo['DESEMPLEADOS'] / tabla_desempleo['PEA']
) * 100

In [0]:
tabla_desempleo.head(100)

In [0]:
# 4. Diccionarios de mapeo
dpto_nombre = {
    5: 'ANTIOQUIA', 8: 'ATLANTICO', 11: 'BOGOTA', 13: 'BOLIVAR', 15: 'BOYACA',
    17: 'CALDAS', 18: 'CAQUETA', 19: 'CAUCA', 20: 'CESAR', 23: 'CORDOBA',
    25: 'CUNDINAMARCA', 27: 'CHOCO', 41: 'HUILA', 44: 'GUAJIRA', 47: 'MAGDALENA',
    50: 'META', 52: 'NARIÑO', 54: 'NORTE DE SANTANDER', 63: 'QUINDIO',
    66: 'RISARALDA', 68: 'SANTANDER', 70: 'SUCRE', 73: 'TOLIMA',
    76: 'VALLE DEL CAUCA', 81: 'ARAUCA', 85: 'CASANARE', 86: 'PUTUMAYO',
    88: 'SAN ANDRES', 91: 'AMAZONAS', 94: 'GUAINIA', 95: 'GUAVIARE',
    97: 'VAUPES', 99: 'VICHADA'
}

sexo = {1: 'Hombre', 2: 'Mujer'}

In [0]:
tabla_desempleo = tabla_desempleo.reset_index()

tabla_desempleo['Departamento'] = tabla_desempleo['DPTO'].map(dpto_nombre)
tabla_desempleo['Sexo'] = tabla_desempleo['P3271'].map(sexo)

In [0]:
tabla_desempleo = tabla_desempleo[
    ['Departamento', 'Sexo', 'PEA', 'DESEMPLEADOS', 'TD']
]


In [0]:
# Ordenar por dos variables de mayor a menor (descendente)
tabla_desempleo = tabla_desempleo.sort_values(
    by=['Sexo', 'TD'],
    ascending=False
)

tabla_desempleo

In [0]:
tabla_desempleo[tabla_desempleo['Sexo'] == 'Mujer']

In [0]:
tabla_desempleo[tabla_desempleo['Sexo'] == 'Hombre']

## Conclusión — Tasas de desempleo por departamento y género, Junio 2026

El análisis de las tasas de desempleo por departamento revela diferencias significativas tanto geográficas como de género. En términos generales, las mujeres presentan tasas de desempleo consistentemente más altas que los hombres en la mayoría de los departamentos.

### Departamentos con mayores tasas de desempleo

| Departamento | Mujeres (TD) | Hombres (TD) |
| --- | --- | --- |
| Putumayo | 25.02% | 19.68% |
| San Andrés | 18.16% | 20.98% |
| Sucre | 20.49% | 6.32% |
| Vichada | 18.44% | 13.23% |
| Bolívar | 17.40% | 6.44% |

Putumayo y San Andrés destacan por presentar las tasas más altas para ambos géneros, coincidiendo con departamentos periféricos y de menor desarrollo relativo. En el caso de las mujeres, Sucre (20.49%) y Bolívar (17.40%) también muestran tasas elevadas, mientras que para los hombres, Arauca (14.49%) y Guaviare (12.13%) se ubican entre los más altos.

### Departamentos con menores tasas de desempleo

| Departamento | Mujeres (TD) | Hombres (TD) |
| --- | --- | --- |
| Vaupés | 1.78% | 2.99% |
| Amazonas | 2.61% | 7.63% |
| Huila | 6.58% | 4.97% |
| Nariño | 6.79% | 4.76% |
| Risaralda | 9.60% | 4.00% |

Vaupés presenta la tasa más baja para ambos géneros, seguido por Amazonas. Huila y Nariño también muestran tasas relativamente bajas en ambos grupos.

### 3. Tomando en consideración únicamente los siguientes grupos de edad: 
### 
### - 18 a 24 
### - 24 a 30 
### - 30 a 40 
### - 40 a 50  
### - 60 en adelante  
### 
### Para cada uno de los departamentos ¿cual es el grupo de edad con mayor y menor tasa de desempleo ?  

In [0]:
# Función para mapear la edad (P6040) a grupos de edad
def mapear_grupo_edad(edad):
    if 18 <= edad < 24:
        return '18 a 24'
    elif 24 <= edad < 30:
        return '24 a 30'
    elif 30 <= edad < 40:
        return '30 a 40'
    elif 40 <= edad < 50:
        return '40 a 50'
    elif edad >= 60:
        return '60 en adelante'
    else:
        return None

# Aplicar el mapeo a ambos DataFrames
desempleados['GRUPO_EDAD'] = desempleados['P6040'].map(mapear_grupo_edad)
ft['GRUPO_EDAD'] = ft['P6040'].map(mapear_grupo_edad)

In [0]:
# Desempleados por grupo de edad y departamento (reutilizando código de pregunta 2)
desempleados_dpto_edad = desempleados[desempleados['DSI'] == 1].groupby(['GRUPO_EDAD', 'DPTO'])['FEX_C18'].sum()

# PEA por grupo de edad y departamento
pea_dpto_edad = ft[ft['FT'] == 1].groupby(['GRUPO_EDAD', 'DPTO'])['FEX_C18'].sum()

# Crear tabla de desempleo (reutilizando estructura de pregunta 2)
tabla_desempleo_edad = pd.DataFrame({'PEA': pea_dpto_edad, 'DESEMPLEADOS': desempleados_dpto_edad})

In [0]:
# Calcular tasa de desempleo (reutilizando código de pregunta 2)
tabla_desempleo_edad['TD'] = (tabla_desempleo_edad['DESEMPLEADOS'] / tabla_desempleo_edad['PEA']) * 100

# Mapear nombres de departamentos (reutilizando dpto_nombre de pregunta 2)
tabla_desempleo_edad = tabla_desempleo_edad.reset_index()
tabla_desempleo_edad['Departamento'] = tabla_desempleo_edad['DPTO'].map(dpto_nombre)

# Seleccionar y ordenar columnas
tabla_desempleo_edad = tabla_desempleo_edad[['Departamento', 'GRUPO_EDAD', 'PEA', 'DESEMPLEADOS', 'TD']]
tabla_desempleo_edad = tabla_desempleo_edad.sort_values(by=['Departamento', 'TD'], ascending=[True, False])

display(tabla_desempleo_edad)

In [0]:
# Grupo de edad con mayor tasa de desempleo por departamento
max_td = tabla_desempleo_edad.loc[tabla_desempleo_edad.groupby('Departamento')['TD'].idxmax()]
max_td = max_td[['Departamento', 'GRUPO_EDAD', 'TD']].reset_index(drop=True)
max_td.columns = ['Departamento', 'Mayor grupo de edad', 'Mayor TD']

# Grupo de edad con menor tasa de desempleo por departamento
min_td = tabla_desempleo_edad.loc[tabla_desempleo_edad.groupby('Departamento')['TD'].idxmin()]
min_td = min_td[['Departamento', 'GRUPO_EDAD', 'TD']].reset_index(drop=True)
min_td.columns = ['Departamento', 'Menor grupo de edad', 'Menor TD']

# Combinar resultados
resultado_edad = pd.merge(max_td, min_td, on='Departamento')
display(resultado_edad)

## Conclusión — Tasa de desempleo por grupo de edad y departamento, Junio 2026

El análisis por grupo de edad revela un patrón claro: los jóvenes de **18 a 24 años** concentran las mayores tasas de desempleo en la mayoría de los departamentos (25 de 33), mientras que las personas de **60 años en adelante** presentan las menores tasas en 23 de los 33 departamentos.

### Departamentos con mayores tasas de desempleo por grupo de edad

| Departamento | Grupo de edad | Mayor TD |
| --- | --- | --- |
| San Andrés | 18 a 24 | 33.66% |
| Sucre | 18 a 24 | 33.14% |
| Vichada | 18 a 24 | 31.60% |
| Putumayo | 24 a 30 | 30.58% |
| Bolívar | 18 a 24 | 30.86% |

San Andrés y Sucre presentan las tasas más altas del país, con más de un 33% de desempleo entre los jóvenes de 18 a 24 años. Esto significa que uno de cada tres jóvenes económicamente activos en estos departamentos está desempleado.

### Departamentos con menores tasas de desempleo por grupo de edad

| Departamento | Grupo de edad | Menor TD |
| --- | --- | --- |
| Risaralda | 60 en adelante | 0.33% |
| Meta | 60 en adelante | 1.18% |
| Chocó | 60 en adelante | 1.21% |
| Cauca | 60 en adelante | 1.27% |
| Magdalena | 60 en adelante | 1.40% |

Las tasas más bajas se concentran en el grupo de 60 años en adelante, con valores inferiores al 1.5% en varios departamentos. Esto se explica porque muchas personas en este rango etario se retiran del mercado laboral (pasan a la población económicamente inactiva), reduciendo el número de desempleados dentro de la PEA.

### Patrones observados

* El grupo de **18 a 24 años** es el más vulnerable al desempleo en el país, coincidiendo con la transición entre educación y el mercado laboral.
* El grupo de **24 a 30 años** presenta las mayores tasas en 7 departamentos (Amazonas, Arauca, Boyacá, Chocó, Nariño, Putumayo y Vaupés), varios de ellos periféricos.
* El grupo de **60 años en adelante** tiene consistentemente las tasas más bajas, posiblemente porque quienes permanecen económicamente activos a esta edad ya tienen trayectorias laborales consolidadas.
* La brecha entre el grupo más afectado y el menos afectado dentro de un mismo departamento puede ser muy amplia: en Bolívar, los jóvenes de 18 a 24 tienen 30.86% de desempleo frente al 2.98% de los mayores de 60, una diferencia de casi 28 puntos porcentuales.